# Error analysis — FP/FN trade-off per run

Loads the predictions saved by `EvalArtifactsCallback` (or regenerates them
from a checkpoint if absent) and produces the figures used in the W&B report:

1. Headline metrics (AUROC / AP / recall @ 1% & 5% FPR)
2. PR + ROC curves
3. Confusion matrices at multiple thresholds
4. Threshold sweep table
5. Per-day error breakdown
6. Top-K worst false positives + false negatives with spectrograms & audio
7. *Optional:* cross-run comparison (overlay PR curves)

Set `RUN_DIR` and `PIPELINE` in the first code cell.


In [ ]:
# --- Parameters ----------------------------------------------------------
RUN_DIR = "outputs/2026-05-23/10-42-17"        # path to a Hydra run directory
PIPELINE = "efficientnet"                       # "efficientnet" or "perch"
TOP_K = 5                                       # number of worst FPs / FNs to show

# Cross-run comparison: list additional run dirs to overlay PR curves with.
COMPARE_RUNS: list[str] = []

# --- Imports --------------------------------------------------------------
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import librosa.display
from IPython.display import Audio, display
from omegaconf import OmegaConf

# Resolve project root + run dir
HERE = Path.cwd()
REPO = HERE if (HERE / "src").exists() else HERE.parent
RUN_DIR = (REPO / RUN_DIR).resolve()
ANALYSIS_DIR = RUN_DIR / "analysis"
print(f"Repo:     {REPO}")
print(f"Run dir:  {RUN_DIR}")
print(f"Pipeline: {PIPELINE}")


## 1. Load or generate predictions

If `predictions.npz` already exists (auto-saved by `EvalArtifactsCallback`),
we load it. Otherwise we reconstruct the model + val DataLoader from the
saved Hydra config in `RUN_DIR/.hydra/config.yaml` and run inference now.

In [ ]:
from narw_classifier.analysis.artifacts import compute_summary, save_run_artifacts
from narw_classifier.analysis.predictions import compute_predictions, load_predictions

PREDS_PATH = ANALYSIS_DIR / "predictions.npz"

if PREDS_PATH.exists():
    print(f"Loading cached predictions from {PREDS_PATH}")
    probs, labels, files = load_predictions(PREDS_PATH)
else:
    import torch

    cfg_path = RUN_DIR / ".hydra" / "config.yaml"
    if not cfg_path.exists():
        raise FileNotFoundError(f"No .hydra config under {RUN_DIR}; cannot regenerate predictions")
    cfg = OmegaConf.load(cfg_path)

    ckpt_path = RUN_DIR / "checkpoints" / "best.ckpt"
    if not ckpt_path.exists():
        ckpt_path = RUN_DIR / "checkpoints" / "last.ckpt"
    print(f"Loading checkpoint: {ckpt_path}")

    if PIPELINE == "efficientnet":
        from narw_classifier.data.datamodule import NARWDataModule
        from narw_classifier.models.baseline import BaselineEfficientNet
        from narw_classifier.models.preprocess import MelImagePreprocessor

        preprocessor = MelImagePreprocessor(
            sample_rate=cfg.preprocess.sample_rate,
            n_fft=cfg.preprocess.n_fft,
            win_length=cfg.preprocess.win_length,
            hop_length=cfg.preprocess.hop_length,
            n_mels=cfg.preprocess.n_mels,
            f_min=cfg.preprocess.f_min,
            f_max=cfg.preprocess.f_max,
            image_size=cfg.preprocess.image_size,
            top_db=cfg.preprocess.top_db,
        )
        datamodule = NARWDataModule(
            data_root=str(REPO / cfg.data.root),
            train_subdir=cfg.data.train_subdir,
            target_sample_rate=cfg.preprocess.sample_rate,
            target_duration_s=cfg.data.target_duration_s,
            val_fraction=cfg.data.val_fraction,
            split_seed=cfg.data.split_seed,
            split_strategy=cfg.data.split_strategy,
            batch_size=cfg.data.batch_size,
            num_workers=0,
            balanced_train_sampler=False,
        )
        model = BaselineEfficientNet.load_from_checkpoint(
            str(ckpt_path), preprocessor=preprocessor, strict=False
        )
    elif PIPELINE == "perch":
        from narw_classifier.data.perch_datamodule import PerchEmbeddingsDataModule
        from narw_classifier.models.perch_linear_probe import PerchLinearProbe

        shift_tag = f"pitch_shift_{int(cfg.preprocess.pitch_shift_semitones)}"
        cache_dir = REPO / cfg.embeddings.cache_dir / cfg.data.split_strategy / shift_tag
        datamodule = PerchEmbeddingsDataModule(
            cache_dir=str(cache_dir),
            batch_size=cfg.data.batch_size,
            num_workers=0,
            balanced_train_sampler=False,
            seed=cfg.data.split_seed,
        )
        model = PerchLinearProbe.load_from_checkpoint(str(ckpt_path))
    else:
        raise ValueError(f"Unknown PIPELINE: {PIPELINE!r}")

    datamodule.prepare_data()
    datamodule.setup()
    val_loader = datamodule.val_dataloader()
    files = getattr(datamodule, "val_files", None) or None

    probs, labels = compute_predictions(model, val_loader, device=torch.device("cpu"))
    print(f"Inferred predictions for {len(probs)} samples")

    save_run_artifacts(ANALYSIS_DIR, probs, labels, files)
    print(f"Saved artifacts to {ANALYSIS_DIR}")

print(f"probs: {probs.shape}  labels: {labels.shape}  positive rate: {labels.mean():.3f}")


## 2. Headline metrics

In [ ]:
summary = compute_summary(probs, labels)
print(json.dumps(summary, indent=2))


## 3. PR + ROC curves

In [ ]:
from narw_classifier.analysis.plots import plot_pr_curve, plot_roc_curve

plot_pr_curve(probs, labels)
plt.show()
plot_roc_curve(probs, labels)
plt.show()


## 4. Confusion matrices across thresholds

Conservation context: false negatives (missed calls) and false positives (false alarms)
have different costs. This block lets you eyeball the trade-off.

In [ ]:
from narw_classifier.analysis.plots import plot_confusion_matrix

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, t in zip(axes, [0.3, 0.5, 0.7, 0.9]):
    f = plot_confusion_matrix(probs, labels, threshold=t)
    img = f.axes[0].images[0]
    plt.close(f)
    ax.imshow(img.get_array(), cmap="Blues")
    cm = img.get_array()
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{int(cm[i, j]):,}", ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], labels=["pred 0", "pred 1"])
    ax.set_yticks([0, 1], labels=["actual 0", "actual 1"])
    ax.set_title(f"threshold = {t:.1f}")
plt.tight_layout()
plt.show()


## 5. Threshold sweep

Precision, recall, FPR, accuracy, and F1 at 21 evenly-spaced thresholds.
Useful for picking an operating point in the report.

In [ ]:
from narw_classifier.analysis.metrics import threshold_sweep

sweep = threshold_sweep(probs, labels)
df = pd.DataFrame(sweep)
df["thresholds"] = df["thresholds"].round(2)
df.round(3)


## 6. Per-day error breakdown

Are errors evenly spread across days, or do specific recording dates dominate?
Requires filenames (saved by the callback).

In [ ]:
from narw_classifier.utils.filenames import parse_filename

if files is None:
    print("No filenames available — skip per-day breakdown.")
else:
    rows = []
    for f, p, y in zip(files, probs, labels):
        parsed = parse_filename(f)
        if parsed is None:
            continue
        rows.append({"date": parsed.date, "prob": float(p), "label": int(y)})
    err_df = pd.DataFrame(rows)
    err_df["pred"] = (err_df["prob"] >= 0.5).astype(int)
    err_df["fp"] = (err_df["pred"] == 1) & (err_df["label"] == 0)
    err_df["fn"] = (err_df["pred"] == 0) & (err_df["label"] == 1)
    by_day = err_df.groupby("date").agg(
        n=("label", "size"),
        n_pos=("label", "sum"),
        n_fp=("fp", "sum"),
        n_fn=("fn", "sum"),
    )
    by_day["fp_rate"] = (by_day["n_fp"] / (by_day["n"] - by_day["n_pos"])).round(3)
    by_day["fn_rate"] = (by_day["n_fn"] / by_day["n_pos"].replace(0, np.nan)).round(3)
    display(by_day.sort_values("n_fp", ascending=False).head(10))


## 7. Worst false positives

Negatives the model called positive with high confidence. These are the
audio clips the model is most "wrong" about in the FP direction.

In [ ]:
def _player(path: Path) -> Audio:
    y, sr = sf.read(str(path))
    # browsers can't decode 2 kHz; resample for the inline player.
    y_play = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=8000)
    return Audio(data=y_play, rate=8000)


def _show_clip(path: Path, prob: float, label: int) -> None:
    print(f"{path.name}    prob={prob:.3f}  label={label}")
    display(_player(path))
    y, sr = sf.read(str(path))
    fig, ax = plt.subplots(figsize=(8, 2))
    S = np.abs(librosa.stft(y.astype(np.float32), n_fft=256, hop_length=64))
    librosa.display.specshow(librosa.amplitude_to_db(S, ref=np.max), sr=sr,
                              hop_length=64, x_axis="time", y_axis="hz", ax=ax, cmap="magma")
    ax.set_ylim(0, 500)
    plt.tight_layout(); plt.show()


train_dir = REPO / "data" / "raw" / "train2"

if files is None:
    print("No filenames available; skip qualitative FP/FN listing.")
else:
    # FPs sorted by prob desc (highest confidence wrong-positives first)
    is_neg = labels == 0
    fp_order = np.argsort(-probs[is_neg])
    fp_files = np.array(files)[is_neg][fp_order][:TOP_K]
    fp_probs = probs[is_neg][fp_order][:TOP_K]
    for fn, p in zip(fp_files, fp_probs):
        _show_clip(train_dir / fn, prob=p, label=0)


## 8. Worst false negatives

Positives the model called negative with high confidence. These are the
upcalls the model misses most badly.

In [ ]:
if files is None:
    print("No filenames available; skip qualitative FP/FN listing.")
else:
    is_pos = labels == 1
    fn_order = np.argsort(probs[is_pos])     # lowest prob among true positives = worst FN
    fn_files = np.array(files)[is_pos][fn_order][:TOP_K]
    fn_probs = probs[is_pos][fn_order][:TOP_K]
    for fn, p in zip(fn_files, fn_probs):
        _show_clip(train_dir / fn, prob=p, label=1)


## 9. Cross-run comparison

Overlays **PR + ROC curves side-by-side** for multiple runs (one figure per metric).

- If `COMPARE_RUNS` in the parameters cell is empty (the default), this **auto-discovers
  every run dir** under `outputs/` that has cached `analysis/predictions.npz` — so once
  you've run `scripts/regenerate_analysis.py` over all your runs, this single cell
  produces the comparison figure for the W&B report.
- Set `COMPARE_RUNS = ["outputs/.../shift0", "outputs/.../shift36"]` to restrict the
  overlay to a specific subset.

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

# If COMPARE_RUNS is empty, auto-discover every run dir with cached predictions.
if COMPARE_RUNS:
    run_paths = [(REPO / r).resolve() for r in ([str(RUN_DIR)] + COMPARE_RUNS)]
else:
    run_paths = sorted({p.parent.parent for p in REPO.glob("outputs/*/*/analysis/predictions.npz")})

curves = []
for run in run_paths:
    preds = run / "analysis" / "predictions.npz"
    if not preds.exists():
        print(f"skip {run.name} (no analysis/predictions.npz)")
        continue
    p, y, _ = load_predictions(preds)
    curves.append((run.name, p, y))

print(f"overlaying {len(curves)} runs")

if curves:
    fig, (ax_pr, ax_roc) = plt.subplots(1, 2, figsize=(13, 5))

    for name, p, y in curves:
        precision, recall, _ = precision_recall_curve(y, p)
        ap = average_precision_score(y, p)
        ax_pr.plot(recall, precision, lw=1.4, label=f"{name}  (AP={ap:.3f})")

        fpr, tpr, _ = roc_curve(y, p)
        auc = roc_auc_score(y, p)
        ax_roc.plot(fpr, tpr, lw=1.4, label=f"{name}  (AUROC={auc:.3f})")

    ax_pr.set(xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.02),
              title="PR curve overlay")
    ax_pr.grid(True, alpha=0.3)
    ax_pr.legend(loc="lower left", fontsize=8)

    ax_roc.plot([0, 1], [0, 1], ls="--", color="grey", lw=0.8, label="random")
    ax_roc.set(xlabel="FPR", ylabel="Recall (TPR)", xlim=(0, 1), ylim=(0, 1.02),
               title="ROC curve overlay")
    ax_roc.grid(True, alpha=0.3)
    ax_roc.legend(loc="lower right", fontsize=8)

    plt.tight_layout()
    plt.show()